# 03. Coleta: população (IBGE)

Baixa a população de cada município do Nordeste, ano a ano (2000 a 2026), e grava em `data/raw/ibge/`.

Não depende da coleta da dengue: pode rodar antes, depois ou ao mesmo tempo. As duas fontes só se juntam depois,
pelo código do município.

| Saída | Conteúdo |
|---|---|
| `data/raw/ibge/populacao_municipios_ne.csv` | todos os municípios e anos num arquivo só |
| `data/raw/ibge/{UF}/{município}.csv` | um arquivo por município (ex.: `RN/Natal.csv`) |

**Neste notebook**
1. De onde vêm os dados
2. Coleta
3. Conferência

In [ ]:
import sys
from pathlib import Path
RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "config.py").exists())
sys.path.append(str(RAIZ / "src"))

import pandas as pd
from coleta.ibge import coleta_ibge as ibge
from config import ANO_INICIO, ANO_FIM_COLETA, N_MUNICIPIOS_NE

## 1. De onde vêm os dados

API SIDRA do IBGE (pública, sem chave). Cada tabela do site https://sidra.ibge.gov.br tem um botão "API" que mostra
o endereço. Exemplo, população de Natal em 2024:
```bash
curl 'https://apisidra.ibge.gov.br/values/t/6579/n6/2408102/v/9324/p/2024'
```

A população de todos os anos precisa de três tabelas, porque os anos de Censo não têm estimativa:

| Tabela | Conteúdo | Anos |
|---|---|---|
| 200 | Censo | 2000 e 2010 |
| 4714 | Censo | 2022 |
| 6579 | Estimativa anual | os demais, exceto 2007 e 2023 |

- Quando um ano tem Censo e estimativa, vale o Censo (é contagem, não projeção).
- **2007 e 2023 não existem** em nenhuma tabela: são preenchidos depois, no `04_carga_ibge.ipynb`.
- Alguns municípios foram criados depois de 2000, por isso os primeiros anos têm um pouco menos de 1794 municípios.

## 2. Coleta
Baixa as três tabelas, junta tudo e grava os arquivos. Leva poucos minutos.
Se o arquivo já existe, não baixa de novo (para baixar outra vez, use `RECOLETAR = True`).

In [ ]:
ARQUIVO = ibge.DESTINO / "populacao_municipios_ne.csv"
RECOLETAR = False

if RECOLETAR or not ARQUIVO.exists():
    ibge.main()
else:
    print(f"Já existe: {ARQUIVO}")

## 3. Conferência
Uma linha por município e ano, com as colunas:

| Coluna | O que é |
|---|---|
| `cod_mun7` | código do município no IBGE (7 dígitos) |
| `cod_mun6` | mesmo código sem o último dígito: é ele que liga com a dengue |
| `uf_cod` | código do estado (21 = MA ... 29 = BA) |
| `municipio` | nome do município |
| `ano` | ano |
| `populacao` | número de habitantes |
| `fonte` | Censo ou estimativa |

In [ ]:
pop = pd.read_csv(ARQUIVO, dtype={"cod_mun7": str, "cod_mun6": str})
print(pop.shape)
pop.head()

### 3.1 Municípios por ano
Quantos municípios têm população em cada ano e de qual tabela veio o dado.

In [ ]:
cobertura = ibge.cobertura(pop).join(pop.groupby("ano")["fonte"].first())
faltando = sorted(set(range(ANO_INICIO, ANO_FIM_COLETA + 1)) - set(pop["ano"]))
print("Anos sem dado (esperado 2007 e 2023):", faltando)
print(f"Anos com menos de {N_MUNICIPIOS_NE} municípios:", list(cobertura.index[cobertura["municipios"] < N_MUNICIPIOS_NE]))
cobertura

### 3.2 População por estado
Soma dos municípios de cada estado, para conferir se os números fazem sentido (ex.: RN ~3,3 milhões, BA ~14 milhões).

In [ ]:
por_uf = pop.pivot_table(index="ano", columns="uf_cod", values="populacao", aggfunc="sum")
por_uf